In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
!pip install langchain-openai

In [ ]:
!pip install -U langchain-chroma

In [ ]:
import os

# 여기에 실제 OpenAI API 키를 입력하세요
os.environ["OPENAI_API_KEY"] = ""


In [ ]:
examples = [
    {
        "instruction": "당신은 IT 프로젝트 관리 전문가입니다. 주어진 상황을 바탕으로 주간 프로젝트 상태 보고서를 작성해 주세요.",
        "input": "2024년 3월 10일, 클라우드 마이그레이션 프로젝트의 주간 점검 회의가 오전 10시에 진행되었다. 프로젝트 매니저 강민호, 시스템 엔지니어 최윤지, 보안 전문가 한승우가 참석했다. 이번 회의의 핵심은 데이터 이전 현황 점검과 보안 취약점 진단 결과에 대한 대응책 마련이었다. 강민호 매니저는 전체 일정에 차질이 없음을 확인했으며, 최윤지 엔지니어는 데이터 이관율 80% 달성을 보고했다.",
        "answer": """
주간 프로젝트 상태 보고서: 클라우드 전환 프로젝트
작성일: 2024년 3월 10일
참석자: 강민호 (PM), 최윤지 (시스템 엔지니어), 한승우 (보안 전문가)

1. 프로젝트 개요
   - 본 회의는 클라우드 마이그레이션 진행 상황 점검을 목적으로 함.
   - 전체적인 마일스톤 대비 진행률은 정상 범주 내에 있음.

2. 기술 진척 현황 (최윤지)
   - 데이터 이관 작업이 현재 80% 완료됨을 보고.
   - 일부 대용량 데이터베이스의 전송 속도 최적화 작업 완료.

3. 보안 및 리스크 관리 (한승우)
   - 보안 취약점 진단 결과 공유 및 대응 전략 논의.
   - 방화벽 설정 업데이트 및 접근 제어 리스트(ACL) 재검토 예정.

4. 향후 계획
   - 다음 주 내로 잔여 데이터 이관(20%) 완료 예정.
   - 최종 마이그레이션 완료 후 사용자 수락 테스트(UAT) 준비 착수.

5. 특이사항
   - 보안 강화 작업으로 인해 다음 주말 중 2시간 내외의 테스트 서버 일시 중단 예정.
""",
    },
    {
        "instruction": "당신은 교육 콘텐츠 요약 전문가입니다. 다음 강의 안내문을 바탕으로 주요 내용을 요약해 주세요.",
        "input": "본 강의는 '생성형 AI의 비즈니스 활용'을 주제로 한 3일 과정의 워크숍입니다. 첫째 날에는 거대언어모델(LLM)의 기본 원리와 프롬프트 엔지니어링의 기초를 배웁니다. 둘째 날에는 기업 내부 데이터를 활용한 RAG 시스템 구축 실습을 진행하며, 마지막 날에는 AI 도입 시 발생할 수 있는 윤리적 문제와 저작권 가이드라인을 다룹니다. 이 워크숍은 실습 위주로 구성되어 있어 참가자들이 직접 결과물을 만들어볼 수 있습니다.",
        "answer": """
강의 요약: 생성형 AI 비즈니스 활용 워크숍

- 교육 목적: 생성형 AI의 원리 이해부터 실무 적용, 윤리적 가이드라인까지 통합적인 실무 역량 강화.
- 커리큘럼 구성:
  1. AI 기초: LLM 원리 및 효과적인 프롬프트 작성법 습득.
  2. 기술 실무: 검색 증강 생성(RAG) 기술을 활용한 기업용 AI 시스템 구축 실습.
  3. 리스크 관리: AI 윤리, 데이터 보안 및 저작권 관련 준수 사항 학습.
- 핵심 특징: 이론 위주의 강의가 아닌 100% 실습 중심 교육으로, 즉각적인 결과물(Output) 도출에 초점.
- 기대 효과: 참가자는 자사 비즈니스 환경에 AI를 안전하고 효율적으로 도입할 수 있는 구체적인 방법론을 체득함.
""",
    },
    {
        "instruction": "당신은 비즈니스 커뮤니케이션 전문가입니다. 다음 구어체 문장을 격식 있는 이메일 문체로 교정해 주세요.",
        "input": "우리가 지난번에 말했던 협력 프로젝트 말인데요, 이번 달 말까지 제안서 보내주시면 좋겠어요. 그래야 저희가 내부적으로 검토해볼 수 있거든요.",
        "answer": "귀사와 논의 중인 협력 프로젝트와 관련하여, 제안서를 이번 달 말까지 송부해 주시기를 요청드립니다. 제안서를 기한 내에 전달해 주시면, 해당 내용을 바탕으로 신속하게 내부 검토를 진행하도록 하겠습니다.",
    },
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import (
    SemanticSimilarityExampleSelector
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma("fewshot_chat", OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}")
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    chroma,
    k=1,
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
)

In [ ]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2024년 2월 15일, 글로벌 솔루션즈의 데이터 플랫폼 팀은 차세대 AI 통합 대시보드 구축을 위한 리스크 점검 회의를 소집했다. 회의에는 기술 이사인 박서준, 백엔드 개발 리드인 정다운, 데이터 엔지니어인 최유리가 참석하였다. 이번 회의에서는 서버 부하 테스트 중 발견된 지연 시간(Latency) 문제의 원인을 분석하고, 최종 배포 전 보안 검수 일정을 재조율하는 것이 핵심 과제였다. 박서준 이사의 주도로 각 파트별 해결 방안이 논의되었으며, 팀은 서비스 안정화 기능을 최우선으로 개발하기로 합의했다."
}

example_selector.select_examples(question)

[{'input': '2024년 3월 10일, 클라우드 마이그레이션 프로젝트의 주간 점검 회의가 오전 10시에 진행되었다. 프로젝트 매니저 강민호, 시스템 엔지니어 최윤지, 보안 전문가 한승우가 참석했다. 이번 회의의 핵심은 데이터 이전 현황 점검과 보안 취약점 진단 결과에 대한 대응책 마련이었다. 강민호 매니저는 전체 일정에 차질이 없음을 확인했으며, 최윤지 엔지니어는 데이터 이관율 80% 달성을 보고했다.',
  'instruction': '당신은 IT 프로젝트 관리 전문가입니다. 주어진 상황을 바탕으로 주간 프로젝트 상태 보고서를 작성해 주세요.',
  'answer': '\n주간 프로젝트 상태 보고서: 클라우드 전환 프로젝트\n작성일: 2024년 3월 10일\n참석자: 강민호 (PM), 최윤지 (시스템 엔지니어), 한승우 (보안 전문가)\n\n1. 프로젝트 개요\n   - 본 회의는 클라우드 마이그레이션 진행 상황 점검을 목적으로 함.\n   - 전체적인 마일스톤 대비 진행률은 정상 범주 내에 있음.\n\n2. 기술 진척 현황 (최윤지)\n   - 데이터 이관 작업이 현재 80% 완료됨을 보고.\n   - 일부 대용량 데이터베이스의 전송 속도 최적화 작업 완료.\n\n3. 보안 및 리스크 관리 (한승우)\n   - 보안 취약점 진단 결과 공유 및 대응 전략 논의.\n   - 방화벽 설정 업데이트 및 접근 제어 리스트(ACL) 재검토 예정.\n\n4. 향후 계획\n   - 다음 주 내로 잔여 데이터 이관(20%) 완료 예정.\n   - 최종 마이그레이션 완료 후 사용자 수락 테스트(UAT) 준비 착수.\n\n5. 특이사항\n   - 보안 강화 작업으로 인해 다음 주말 중 2시간 내외의 테스트 서버 일시 중단 예정.\n'}]

In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.",
        ),
        few_shot_prompt,
        ("human", "{instruction}\n{input}")
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    max_tokens=2048,
    temperature=0.1,
)

In [ ]:
chain = final_prompt | llm
chain.invoke(question).content

'회의록\n\n회의명: 차세대 AI 통합 대시보드 구축 리스크 점검 회의\n일시: 2024년 2월 15일\n장소: 글로벌 솔루션즈 본사 회의실\n참석자: 박서준 (기술 이사), 정다운 (백엔드 개발 리드), 최유리 (데이터 엔지니어)\n\n1. 개회\n   - 박서준 이사가 회의를 주재하며 회의 목적 및 주요 안건 설명.\n\n2. 서버 부하 테스트 결과 분석\n   - 정다운 리드가 서버 부하 테스트 중 발견된 지연 시간 문제에 대해 보고.\n   - 지연 시간의 주요 원인으로 데이터 처리 병목 현상 및 네트워크 대역폭 제한 식별.\n\n3. 해결 방안 논의\n   - 최유리 엔지니어가 데이터 처리 최적화 방안 제안.\n   - 네트워크 대역폭 확장을 위한 인프라 업그레이드 필요성 검토.\n   - 박서준 이사가 서비스 안정화 기능을 최우선으로 개발할 것을 제안, 참석자 전원 동의.\n\n4. 보안 검수 일정 재조율\n   - 최종 배포 전 보안 검수 일정을 재조정하기로 결정.\n   - 보안 팀과의 협의를 통해 새로운 일정 수립 예정.\n\n5. 기타 논의 사항\n   - 추가적인 리스크 요소 식별 및 대응 방안 마련 필요성 강조.\n   - 다음 회의 일정 및 준비 사항 논의.\n\n6. 폐회\n   - 박서준 이사가 회의를 마무리하며, 각 팀의 협조와 신속한 문제 해결을 당부.\n\n회의 종료: 2024년 2월 15일, 오후 3시\n\n작성자: [작성자 이름]'